# Sprint 2 - Task BF-24: NetworkX Topological Feature Extraction & Baseline Comparison



In [1]:
!pip -q install torch_geometric xgboost
import os, time
import numpy as np
import pandas as pd
import networkx as nx
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
print('torch', torch.__version__, '| networkx', nx.__version__)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.1+cpu | networkx 3.7


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.Path Configuration & Graph Loading from BF-22

In [2]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_PATH = '/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_graph_temporal.pt'
OUT_DIR = '/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'file not found: {CKPT_PATH}')
print(' CKPT_PATH =', CKPT_PATH)
print(' OUT_DIR =', OUT_DIR)

ckpt = torch.load(CKPT_PATH, weights_only=False)
data = ckpt['data']
TRAIN_RANGE, TEST_RANGE = ckpt['train_range'], ckpt['test_range']
N = data.num_nodes
print(data)
print('Train range:', TRAIN_RANGE, '| Test range:', TEST_RANGE)


Mounted at /content/drive
 CKPT_PATH = /content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_graph_temporal.pt
 OUT_DIR = /content/drive/MyDrive/Explainable-AML-GAT-ResNet/data
Data(x=[203769, 166], edge_index=[2, 234355], y=[203769], time_step=[203769], train_mask=[203769], test_mask=[203769], train_labeled_mask=[203769], test_labeled_mask=[203769])
Train range: (1, 34) | Test range: (35, 49)


In [4]:
N_FEAT = 166

def get_feature_matrix(x, y, n_feat=N_FEAT):
    """Ensure exact n_feat columns (float32). Handles 169-col layout from BF-22."""
    x = x.numpy() if torch.is_tensor(x) else np.asarray(x)
    if x.shape[1] == n_feat:
        return x.astype(np.float32)
    if x.shape[1] == n_feat + 3:
        first_is_idx = np.array_equal(x[:, 0].astype(np.int64), np.arange(len(x)))
        last_is_y    = np.array_equal(x[:, -1].astype(np.int64), y)
        if first_is_idx and last_is_y:
            print(f'[!] data.x contains {x.shape[1]} columns: col 0 = node_idx, last col = label y (leakage) '
                  f'-> Sliced back to {n_feat} feature columns.')
            return x[:, 1:1 + n_feat].astype(np.float32)
        raise ValueError(f'data.x has {x.shape[1]} columns but layout does not match expected structure.')
    raise ValueError(f'data.x has {x.shape[1]} columns, expected {n_feat}.')

ts = data.time_step.numpy().astype(int)
ei = data.edge_index.numpy()
y_np = data.y.numpy()
cross = int((ts[ei[0]] != ts[ei[1]]).sum())

print(f'N = {N} | E = {ei.shape[1]} | time_step {ts.min()}..{ts.max()}')
print('Cross time-step edges:', cross, '(expected: 0)')
print('Self-loops:', int((ei[0] == ei[1]).sum()))
assert N == 203769, f"Node count mismatch! Expected 203769, got {N}"

X166 = get_feature_matrix(data.x, y_np)
assert X166.shape == (N, 166)
print(f'Feature matrix shape: {X166.shape}')

_m = data.train_labeled_mask.numpy()
_corr = np.nan_to_num(np.array([np.corrcoef(X166[_m, j], y_np[_m])[0, 1] for j in range(N_FEAT)]))
print(f'Max |corr(feature, y)| = {np.abs(_corr).max():.3f} (cột {int(np.abs(_corr).argmax())})')
assert np.abs(_corr).max() < 0.95, 'Có cột đặc trưng gần như trùng nhãn -> nghi rò rỉ!'

N = 203769 | E = 234355 | time_step 1..49
Cross time-step edges: 0 (expected: 0)
Self-loops: 0
Feature matrix shape: (203769, 166)
Max |corr(feature, y)| = 0.307 (cột 53)


## 2. Extract 5 Topologica
For each time step `t`, we build an `nx.DiGraph` containing only the nodes and edges from snapshot `t`, then compute:

| Feature | AML |
|---|---|
| `in_degree` |Number of incoming transaction flows to the address|
| `out_degree` | Number of outgoing transaction flows from the address |
| `in_out_ratio = in / (out + ε)` | Cash flow imbalance (funds consolidation / splitting pattern) |
| `pagerank` (α=0.85) | Importance and influence score of the node within the network |
| `clustering` | Level of connectivity among the node's neighbors (computed on undirected graph) |

Calculations are done per time step snapshot (not on the full graph) so that PageRank values are properly normalized within each time period.

In [5]:
TOPO_NAMES = ['in_degree', 'out_degree', 'in_out_ratio', 'pagerank', 'clustering']
EPS = 1e-6

def extract_topo_features(edge_index, time_step, num_nodes, eps=EPS, alpha=0.85, verbose=True):
    """edge_index: np (2,E) | time_step: np (N,) | return: np float32 (N,5) theo TOPO_NAMES"""
    time_step = np.asarray(time_step).astype(int)
    src, dst = edge_index[0], edge_index[1]
    X = np.zeros((num_nodes, 5), dtype=np.float32)
    for t in np.unique(time_step):
        t0 = time.time()
        nodes = np.where(time_step == t)[0]
        m = (time_step[src] == t) & (time_step[dst] == t)
        G = nx.DiGraph()
        G.add_nodes_from(nodes.tolist())
        G.add_edges_from(zip(src[m].tolist(), dst[m].tolist()))
        in_deg, out_deg = dict(G.in_degree()), dict(G.out_degree())
        pr  = nx.pagerank(G, alpha=alpha)
        clu = nx.clustering(G.to_undirected())
        for n in nodes.tolist():
            X[n] = (in_deg[n], out_deg[n], in_deg[n] / (out_deg[n] + eps), pr[n], clu[n])
        if verbose:
            print(f'  t={t:2d}: {len(nodes):5d} nodes, {int(m.sum()):5d} edges ({time.time()-t0:.1f}s)')
    return X

In [6]:
t0 = time.time()
X_topo = extract_topo_features(ei, ts, N)
print(f'Done in {time.time()-t0:.0f}s')

assert X_topo.shape == (N, 5), X_topo.shape
assert not np.isnan(X_topo).any() and not np.isinf(X_topo).any()
print('X_topo:', X_topo.shape)

  t= 1:  7880 nodes,  9164 edges (0.2s)
  t= 2:  4544 nodes,  5241 edges (0.1s)
  t= 3:  6621 nodes,  8316 edges (0.2s)
  t= 4:  5693 nodes,  8180 edges (0.2s)
  t= 5:  6803 nodes,  8623 edges (0.5s)
  t= 6:  4328 nodes,  5242 edges (0.2s)
  t= 7:  6048 nodes,  7253 edges (0.3s)
  t= 8:  4457 nodes,  5186 edges (0.2s)
  t= 9:  4996 nodes,  5939 edges (0.1s)
  t=10:  6727 nodes,  8588 edges (0.2s)
  t=11:  4296 nodes,  4656 edges (0.1s)
  t=12:  2047 nodes,  2213 edges (0.1s)
  t=13:  4528 nodes,  4827 edges (0.1s)
  t=14:  2022 nodes,  2078 edges (0.0s)
  t=15:  3639 nodes,  3823 edges (0.1s)
  t=16:  2975 nodes,  3120 edges (0.1s)
  t=17:  3385 nodes,  3650 edges (0.1s)
  t=18:  1976 nodes,  2115 edges (0.0s)
  t=19:  3506 nodes,  3838 edges (0.1s)
  t=20:  4291 nodes,  4755 edges (0.1s)
  t=21:  3537 nodes,  3959 edges (0.1s)
  t=22:  5894 nodes,  7014 edges (0.2s)
  t=23:  4165 nodes,  4584 edges (0.1s)
  t=24:  4592 nodes,  5124 edges (0.1s)
  t=25:  2314 nodes,  2619 edges (0.1s)


## 3. Exploratory Data Analysis & Class Comparison

In [7]:
df_topo = pd.DataFrame(X_topo, columns=TOPO_NAMES)
df_topo['y'] = data.y.numpy()
display(df_topo[TOPO_NAMES].describe().T)

lab = df_topo[df_topo.y != -1].copy()
lab['class'] = lab.y.map({1: 'illicit', 0: 'licit'})
display(lab.groupby('class')[TOPO_NAMES].mean().T)

,count,mean,std,min,25%,50%,75%,max
in_degree,203769.0,1.150101,3.909707e+00,0.000000,0.000000,1.000000,1.000000,2.840000e+02
out_degree,203769.0,1.150101,1.893592e+00,0.000000,1.000000,1.000000,1.000000,4.720000e+02
in_out_ratio,203769.0,379229.062500,3.134244e+06,0.000000,0.000000,0.999999,0.999999,2.840000e+08
pagerank,203769.0,0.000240,4.628722e-04,0.000036,0.000086,0.000146,0.000264,2.655411e-02
clustering,203769.0,0.013762,9.741712e-02,0.000000,0.000000,0.000000,0.000000,1.000000e+00


class,illicit,licit
in_degree,1.269967,1.909398
out_degree,0.741694,1.185821
in_out_ratio,808581.312500,835004.062500
pagerank,0.000222,0.000272
clustering,0.000356,0.008320


## 4. Feature Scaling — Data Leakage Prevention


In [8]:
def scale_topo(X, train_mask):
    Z = X.astype(np.float64).copy()
    Z[:, :3] = np.log1p(Z[:, :3])
    Z[:, 3]  = np.log1p(Z[:, 3] * 1e4)
    sc = StandardScaler().fit(Z[train_mask])
    return sc.transform(Z).astype(np.float32), sc

train_mask = data.train_mask.numpy()
X_topo_scaled, topo_scaler = scale_topo(X_topo, train_mask)
print('Mean (train):', X_topo_scaled[train_mask].mean(0).round(3))
print('Std  (train):', X_topo_scaled[train_mask].std(0).round(3))

Mean (train): [ 0. -0. -0.  0.  0.]
Std  (train): [1. 1. 1. 1. 1.]


## 5.Save Outputs

In [9]:
topo_out = {'node_idx': torch.arange(N),
            'x_topo': torch.from_numpy(X_topo),
            'x_topo_scaled': torch.from_numpy(X_topo_scaled),
            'names': TOPO_NAMES,
            'scaler_mean': topo_scaler.mean_, 'scaler_scale': topo_scaler.scale_}
pt_path = os.path.join(OUT_DIR, 'elliptic_topo_features.pt')
torch.save(topo_out, pt_path)

out_df = pd.DataFrame(X_topo, columns=TOPO_NAMES)
out_df.insert(0, 'node_idx', np.arange(N))
for j, nme in enumerate(TOPO_NAMES):
    out_df[nme + '_scaled'] = X_topo_scaled[:, j]
csv_path = os.path.join(OUT_DIR, 'elliptic_topo_features.csv')
out_df.to_csv(csv_path, index=False)
print('Saved:', pt_path, f'({os.path.getsize(pt_path)/1e6:.1f} MB)')
print('Saved:', csv_path, f'({os.path.getsize(csv_path)/1e6:.1f} MB)')

Saved: /content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_topo_features.pt (9.8 MB)
Saved: /content/drive/MyDrive/Explainable-AML-GAT-ResNet/data/elliptic_topo_features.csv (19.5 MB)


## 6.Performance Evaluation: Baseline 1 vs 166+Topo vs GAT-ResNet


In [10]:
train_idx = torch.nonzero(data.train_labeled_mask).view(-1).numpy()
test_idx  = torch.nonzero(data.test_labeled_mask).view(-1).numpy()
y_np = data.y.numpy()
y_tr, y_te = y_np[train_idx], y_np[test_idx]
print(f'Train labeled: {len(train_idx)} (illicit {int(y_tr.sum())}) | Test labeled: {len(test_idx)} (illicit {int(y_te.sum())})')

def calc_metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    return {'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall':    recall_score(y_true, pred, zero_division=0),
            'F1_illicit': f1_score(y_true, pred, zero_division=0),
            'PR_AUC':    average_precision_score(y_true, prob),
            'ROC_AUC':   roc_auc_score(y_true, prob)}

def make_models(y_train, seed=42):
    models = {'RandomForest': RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed,
                                                     class_weight='balanced_subsample')}
    try:
        from xgboost import XGBClassifier
        spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        models['XGBoost'] = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                                          colsample_bytree=0.8, scale_pos_weight=spw, eval_metric='logloss',
                                          n_jobs=-1, random_state=seed)
    except ImportError:
        print('XGBoost not installed -> Running RandomForest only')
    return models

def evaluate_feature_sets(feature_sets, tr_idx, te_idx, y_all):
    rows = []
    for fs_name, X in feature_sets.items():
        for m_name, model in make_models(y_all[tr_idx]).items():
            model.fit(X[tr_idx], y_all[tr_idx])
            prob = model.predict_proba(X[te_idx])[:, 1]
            rows.append({'Features': fs_name, 'Model': m_name, '#Feat': X.shape[1],
                         **calc_metrics(y_all[te_idx], prob)})
            print('done:', fs_name, '/', m_name)
    return pd.DataFrame(rows)

Train labeled: 29894 (illicit 3462) | Test labeled: 16670 (illicit 1083)


In [11]:
x_np = X166
feature_sets = {'Baseline 1 (166)': x_np,
                '166 + Topo (171)': np.hstack([x_np, X_topo_scaled])}
results = evaluate_feature_sets(feature_sets, train_idx, test_idx, y_np)
display(results.round(4))

done: Baseline 1 (166) / RandomForest
done: Baseline 1 (166) / XGBoost
done: 166 + Topo (171) / RandomForest
done: 166 + Topo (171) / XGBoost


,Features,Model,#Feat,Precision,Recall,F1_illicit,PR_AUC,ROC_AUC
0,Baseline 1 (166),RandomForest,166,0.9893,0.6842,0.8090,0.7943,0.9310
1,Baseline 1 (166),XGBoost,166,0.8641,0.7341,0.7938,0.8012,0.9298
2,166 + Topo (171),RandomForest,171,0.9896,0.6999,0.8199,0.7952,0.9338
3,166 + Topo (171),XGBoost,171,0.8843,0.7341,0.8022,0.7978,0.9203


### GAT-ResNet Integration Check
File `gat_resnet_preds.pt` do task huấn luyện GAT-ResNet cung cấp, gồm:
- `node_idx`: chỉ số các nút trong `test_labeled_mask`
- `prob_illicit`: xác suất illicit tương ứng từng nút trên
- `embeddings`: tensor [203769, 100] cho toàn bộ nút, theo thứ tự node index


In [12]:
GAT_PRED_PATH = None  # e.g., os.path.join(OUT_DIR, 'gat_resnet_preds.pt')

gat_rows, emb = [], None
if GAT_PRED_PATH and os.path.exists(GAT_PRED_PATH):
    g = torch.load(GAT_PRED_PATH, weights_only=False)
    g_idx, g_prob = g['node_idx'].numpy(), g['prob_illicit'].numpy()
    assert set(g_idx.tolist()) == set(test_idx.tolist()), 'GAT-ResNet test set mismatch!'
    order = pd.Series(np.arange(len(g_idx)), index=g_idx).loc[test_idx].values
    gat_rows.append({
        'Features': 'GAT-ResNet (end-to-end)', 'Model': 'GAT-ResNet', '#Feat': '-',
        **calc_metrics(y_te, g_prob[order])
    })
    emb = g.get('embeddings')
else:
    print('GAT-ResNet results not found -> Skipping (Set GAT_PRED_PATH when Step 3 finishes).')

if emb is not None:
    emb = emb.numpy()
    full = evaluate_feature_sets(
        {'166 + Topo + GAT emb (271)': np.hstack([X166, X_topo_scaled, emb])},
        train_idx, test_idx, y_np
    )
    results = pd.concat([results, full], ignore_index=True)

final = pd.concat([results, pd.DataFrame(gat_rows)], ignore_index=True)
display(final.round(4))
final.to_csv(os.path.join(OUT_DIR, 'comparison_baseline_vs_gat.csv'), index=False)

GAT-ResNet results not found -> Skipping (Set GAT_PRED_PATH when Step 3 finishes).


,Features,Model,#Feat,Precision,Recall,F1_illicit,PR_AUC,ROC_AUC
0,Baseline 1 (166),RandomForest,166,0.9893,0.6842,0.8090,0.7943,0.9310
1,Baseline 1 (166),XGBoost,166,0.8641,0.7341,0.7938,0.8012,0.9298
2,166 + Topo (171),RandomForest,171,0.9896,0.6999,0.8199,0.7952,0.9338
3,166 + Topo (171),XGBoost,171,0.8843,0.7341,0.8022,0.7978,0.9203


## 7. Feature Importance Analysis

In [13]:
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight='balanced_subsample')
X_171 = np.hstack([X166, X_topo_scaled])
rf.fit(X_171[train_idx], y_tr)

names = ['time_step'] + [f'f{i}' for i in range(2, 167)] + TOPO_NAMES
print(f'Total feature names: {len(names)} | Model features: {len(rf.feature_importances_)}')

assert len(names) == len(rf.feature_importances_), "Feature count mismatch!"
imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)

print('\nTop 15 Most Important Features:')
display(imp.head(15).to_frame('importance'))

print('\n Ranks of 5 NetworkX Topological Features:')
for name in TOPO_NAMES:
    rank = list(imp.index).index(name) + 1
    pct = imp[name] * 100
    print(f'   {name:15s} → Rank {rank:3d} / 171 | {pct:.2f}% of total importance')



Total feature names: 171 | Model features: 171

Top 15 Most Important Features:


,importance
f54,0.073999
f48,0.043256
f56,0.043019
f50,0.039296
f30,0.033552
f42,0.032180
f44,0.031024
f6,0.024271
f91,0.023605
f15,0.022653



 Ranks of 5 NetworkX Topological Features:
   in_degree       → Rank 135 / 171 | 0.06% of total importance
   out_degree      → Rank 157 / 171 | 0.03% of total importance
   in_out_ratio    → Rank 139 / 171 | 0.05% of total importance
   pagerank        → Rank  90 / 171 | 0.17% of total importance
   clustering      → Rank 168 / 171 | 0.01% of total importance


In [14]:
rows = []
for seed in [0, 1, 2, 3, 4]:
    for fs_name, X in feature_sets.items():
        for m_name, model in make_models(y_np[train_idx], seed=seed).items():
            model.fit(X[train_idx], y_np[train_idx])
            prob = model.predict_proba(X[test_idx])[:, 1]
            rows.append({'Features': fs_name, 'Model': m_name, 'seed': seed,
                         **calc_metrics(y_np[test_idx], prob)})
multi = pd.DataFrame(rows)
display(multi.groupby(['Features', 'Model'])[['Precision', 'Recall', 'F1_illicit', 'PR_AUC', 'ROC_AUC']]
        .agg(['mean', 'std']).round(4))

Precision          Recall         F1_illicit  \
                                   mean     std    mean     std       mean   
Features         Model                                                       
166 + Topo (171) RandomForest    0.9878  0.0011  0.6898  0.0032     0.8123   
                 XGBoost         0.8664  0.0199  0.7341  0.0017     0.7947   
Baseline 1 (166) RandomForest    0.9894  0.0016  0.6898  0.0028     0.8128   
                 XGBoost         0.8597  0.0051  0.7363  0.0019     0.7932   

                                       PR_AUC         ROC_AUC          
                                  std    mean     std    mean     std  
Features         Model                                                 
166 + Topo (171) RandomForest  0.0021  0.7948  0.0020  0.9336  0.0040  
                 XGBoost       0.0075  0.8000  0.0022  0.9272  0.0041  
Baseline 1 (166) RandomForest  0.0021  0.7954  0.0016  0.9343  0.0026  
                 XGBoost       0.0023  0.8000  0.0014  0.9268  0.0035

In [15]:
multi.to_csv(os.path.join(OUT_DIR, 'comparison_multiseed.csv'), index=False)

In [18]:
from getpass import getpass
import subprocess
import os

# 1. Nhập GitHub Token của bạn vào ô trống hiện ra
token = getpass("New GitHub Token: ")
repo_url = f"https://{token}@github.com/thuyduong955/Explainable-AML-GAT-ResNet.git"

# 2. Clone repo về Colab (nếu chưa có)
if not os.path.exists("/content/Explainable-AML-GAT-ResNet"):
    print("Đang clone repository...")
    subprocess.run(["git", "clone", repo_url, "/content/Explainable-AML-GAT-ResNet"], check=True)

# 3. Di chuyển vào thư mục project
%cd /content/Explainable-AML-GAT-ResNet

# 4. Gắn token vào remote origin để cho phép push code
subprocess.run(["git", "remote", "set-url", "origin", repo_url], check=True)
del token # Xóa token khỏi bộ nhớ để bảo mật

# 5. Cấu hình thông tin người commit
!git config user.email "23521845@gm.uit.edu.vn"
!git config user.name "ThienY173"

# 6. Kiểm tra trạng thái
!git status

New GitHub Token: ··········
Đang clone repository...
/content/Explainable-AML-GAT-ResNet
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [19]:
# 1. Rời khỏi main, tạo và chuyển sang nhánh mới ngay lập tức (bạn có thể đổi tên nhánh nếu muốn)
!git checkout -b feature/BF-24-network-topo

# 2. Copy file notebook của bạn vào thư mục data
!cp "/content/networkx_topo_features (4).ipynb" "/content/Explainable-AML-GAT-ResNet/data/"

# 3. Đưa folder data vào danh sách chuẩn bị commit
!git add data/

# 4. Kiểm tra trạng thái (Để bạn yên tâm là mình đang ở nhánh mới, không phải main)
!git status

# 5. Ghi nhận thay đổi (Commit)
!git commit -m "BF-24: Thêm notebook network-topo vào thư mục data"

# 6. Push thẳng lên nhánh mới trên GitHub
!git push --set-upstream origin feature/BF-24-network-topo

Switched to a new branch 'feature/BF-24-network-topo'
cp: cannot stat '/content/networkx_topo_features (4).ipynb': No such file or directory
On branch feature/BF-24-network-topo
nothing to commit, working tree clean
On branch feature/BF-24-network-topo
nothing to commit, working tree clean
Total 0 (delta 0), reused 0 (delta 0), pack-reused 0
remote: 
remote: Create a pull request for 'feature/BF-24-network-topo' on GitHub by visiting:
remote:      https://github.com/thuyduong955/Explainable-AML-GAT-ResNet/pull/new/feature/BF-24-network-topo
remote: 
To https://github.com/thuyduong955/Explainable-AML-GAT-ResNet.git
 * [new branch]      feature/BF-24-network-topo -> feature/BF-24-network-topo
branch 'feature/BF-24-network-topo' set up to track 'origin/feature/BF-24-network-topo'.
